# Probability Model Validation & Out-of-Sample Testing

This notebook validates the GBM (Geometric Brownian Motion) probability model against actual Polymarket outcomes.

**Goals:**
1. Check if predicted probabilities match realized win rates (calibration)
2. Out-of-sample walk-forward testing (no lookahead bias)
3. Parameter sensitivity without overfitting
4. Volatility regime analysis
5. Actionable recommendations

In [ ]:
import math
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path("data")
MARKETS = {
    "btc_15m": {"subdir": "btc_15m", "window_s": 900, "label": "BTC 15m"},
    "btc_5m":  {"subdir": "btc_5m",  "window_s": 300, "label": "BTC 5m"},
    "eth_15m": {"subdir": "eth_15m", "window_s": 900, "label": "ETH 15m"},
    "eth_5m":  {"subdir": "eth_5m",  "window_s": 300, "label": "ETH 5m"},
}

def norm_cdf(x):
    return 0.5 * math.erfc(-x / math.sqrt(2.0))

def poly_fee(p):
    return 0.25 * (p * (1.0 - p)) ** 2

def compute_vol_deduped(prices, timestamps=None):
    """Realized vol from price series, ignoring stale (duplicate) ticks."""
    changes = []
    for i, p in enumerate(prices):
        ts = timestamps[i] if timestamps is not None else i * 1000
        if p > 0 and (not changes or p != changes[-1][1]):
            changes.append((i, p, ts))
    if len(changes) < 3:
        return 0.0
    log_rets = []
    for j in range(1, len(changes)):
        if timestamps is not None:
            dt = (changes[j][2] - changes[j-1][2]) / 1000.0
        else:
            dt = changes[j][0] - changes[j-1][0]
        if dt > 0:
            lr = math.log(changes[j][1] / changes[j-1][1])
            log_rets.append(lr / math.sqrt(dt))
    if len(log_rets) < 2:
        return 0.0
    return float(np.std(log_rets, ddof=1))

print("Utilities loaded.")

## 1. Load All Window Data

In [ ]:
MIN_FINAL_REMAINING_S = 5.0
MAX_START_GAP_S = 30.0

def load_windows(market_key):
    """Load all complete windows for a market. Returns list of (df, outcome_up, start_px, final_px)."""
    cfg = MARKETS[market_key]
    data_dir = DATA_DIR / cfg["subdir"]
    files = sorted(data_dir.glob("*.parquet"))
    windows = []
    skipped = 0
    
    for f in files:
        df = pd.read_parquet(f)
        if df.empty:
            skipped += 1
            continue
        if "chainlink_btc" in df.columns and "chainlink_price" not in df.columns:
            df.rename(columns={"chainlink_btc": "chainlink_price"}, inplace=True)
        
        # Check completeness
        if "window_end_ms" in df.columns:
            end_ms = df["window_end_ms"].iloc[0]
            last_ts = df["ts_ms"].iloc[-1]
            if last_ts < end_ms:
                skipped += 1
                continue
        else:
            final_remaining = df["time_remaining_s"].iloc[-1]
            if final_remaining > MIN_FINAL_REMAINING_S:
                skipped += 1
                continue
        
        # Skip late-start windows
        if ("window_start_ms" in df.columns and "window_end_ms" in df.columns
                and "time_remaining_s" in df.columns):
            window_dur_s = (df["window_end_ms"].iloc[0] - df["window_start_ms"].iloc[0]) / 1000
            first_remaining = df["time_remaining_s"].iloc[0]
            if first_remaining < window_dur_s - MAX_START_GAP_S:
                skipped += 1
                continue
        
        # Determine outcome
        start_prices = df["window_start_price"].dropna()
        if start_prices.empty:
            skipped += 1
            continue
        start_px = float(start_prices.iloc[0])
        final_px = float(df["chainlink_price"].iloc[-1])
        if pd.isna(start_px) or pd.isna(final_px) or start_px == 0:
            skipped += 1
            continue
        outcome = 1 if final_px >= start_px else 0
        
        windows.append((df, outcome, start_px, final_px))
    
    # Sort by first timestamp
    windows.sort(key=lambda x: x[0]["ts_ms"].iloc[0])
    return windows, skipped

# Load all markets
all_windows = {}
for mk in MARKETS:
    wins, skp = load_windows(mk)
    all_windows[mk] = wins
    up_count = sum(1 for _, o, _, _ in wins if o == 1)
    print(f"{MARKETS[mk]['label']:>8s}: {len(wins):3d} complete windows ({skp} skipped) | "
          f"UP: {up_count} ({up_count/len(wins)*100:.1f}%) | DOWN: {len(wins)-up_count} ({(len(wins)-up_count)/len(wins)*100:.1f}%)")

## 2. GBM Calibration Analysis

For each window, at multiple time points, we compute:
- `z = delta / (sigma * sqrt(tau))`
- `p_model = Phi(z)` (GBM prediction)
- Compare against actual outcome

A well-calibrated model: when we predict 70% UP, about 70% of those windows should actually go UP.

In [ ]:
def extract_signals(windows, vol_lookback_s=90, max_z=1.5, sample_every=30):
    """Extract (p_model, z, sigma, tau, outcome) at regular intervals across all windows."""
    records = []
    for df, outcome, start_px, final_px in windows:
        prices = df["chainlink_price"].tolist()
        ts_list = df["ts_ms"].tolist()
        
        for idx in range(vol_lookback_s, len(df), sample_every):
            row = df.iloc[idx]
            tau = row["time_remaining_s"]
            if tau <= 0:
                continue
            
            lo = max(0, idx - vol_lookback_s)
            price_slice = prices[lo:idx+1]
            ts_slice = ts_list[lo:idx+1]
            
            # Skip gaps
            has_gap = False
            for k in range(1, len(ts_slice)):
                if ts_slice[k] - ts_slice[k-1] > 5000:
                    has_gap = True
                    break
            if has_gap:
                continue
            
            sigma = compute_vol_deduped(price_slice, ts_slice)
            if sigma <= 0:
                continue
            
            current_px = row["chainlink_price"]
            delta = (current_px - start_px) / start_px
            z_raw = delta / (sigma * math.sqrt(tau))
            z_capped = max(-max_z, min(max_z, z_raw))
            p_model = norm_cdf(z_capped)
            
            records.append({
                "p_model": p_model,
                "z_raw": z_raw,
                "z_capped": z_capped,
                "sigma": sigma,
                "tau": tau,
                "delta": delta,
                "outcome_up": outcome,
                "start_px": start_px,
                "current_px": current_px,
                "final_px": final_px,
            })
    
    return pd.DataFrame(records)

# Extract signals for all markets
signal_dfs = {}
for mk in MARKETS:
    sdf = extract_signals(all_windows[mk])
    signal_dfs[mk] = sdf
    print(f"{MARKETS[mk]['label']:>8s}: {len(sdf):,d} signal observations")

In [ ]:
def calibration_analysis(sdf, n_bins=10, label=""):
    """Bin p_model into deciles and compare predicted vs actual."""
    if sdf.empty:
        print(f"  {label}: No data")
        return None
    
    sdf = sdf.copy()
    sdf["p_bin"] = pd.cut(sdf["p_model"], bins=n_bins, labels=False)
    
    rows = []
    for b in range(n_bins):
        mask = sdf["p_bin"] == b
        if mask.sum() == 0:
            continue
        subset = sdf[mask]
        pred_mean = subset["p_model"].mean()
        actual_mean = subset["outcome_up"].mean()
        n = len(subset)
        # 95% CI for actual rate
        se = math.sqrt(actual_mean * (1 - actual_mean) / n) if n > 1 else 0
        rows.append({
            "bin": b,
            "pred_p": round(pred_mean, 3),
            "actual_p": round(actual_mean, 3),
            "gap": round(actual_mean - pred_mean, 3),
            "n_obs": n,
            "ci_95_lo": round(max(0, actual_mean - 1.96*se), 3),
            "ci_95_hi": round(min(1, actual_mean + 1.96*se), 3),
        })
    
    cal_df = pd.DataFrame(rows)
    
    # Brier score (lower = better, 0.25 = random)
    brier = ((sdf["p_model"] - sdf["outcome_up"]) ** 2).mean()
    # ECE (Expected Calibration Error)
    ece = sum(abs(r["gap"]) * r["n_obs"] for _, r in cal_df.iterrows()) / len(sdf)
    
    print(f"\n{'='*70}")
    print(f"  CALIBRATION: {label}")
    print(f"  Brier Score: {brier:.4f} (0.25=random, 0=perfect)")
    print(f"  ECE:         {ece:.4f} (0=perfect calibration)")
    print(f"{'='*70}")
    print(cal_df.to_string(index=False))
    
    return cal_df, brier, ece

# Run calibration for each market
cal_results = {}
for mk in MARKETS:
    result = calibration_analysis(signal_dfs[mk], label=MARKETS[mk]["label"])
    if result is not None:
        cal_results[mk] = result

## 3. Z-Score Distribution & Outcome Analysis

Look at how z-scores distribute and whether extreme z actually predicts outcomes better.

In [ ]:
def z_score_analysis(sdf, label=""):
    """Analyze win rates by z-score bins."""
    if sdf.empty:
        return
    
    # Z-bin analysis
    z_edges = [-3, -2, -1.5, -1, -0.5, 0, 0.5, 1, 1.5, 2, 3]
    sdf = sdf.copy()
    sdf["z_bin"] = pd.cut(sdf["z_capped"], bins=z_edges)
    
    print(f"\n{'='*70}")
    print(f"  Z-SCORE ANALYSIS: {label}")
    print(f"{'='*70}")
    
    rows = []
    for zb in sdf["z_bin"].cat.categories:
        mask = sdf["z_bin"] == zb
        if mask.sum() == 0:
            continue
        sub = sdf[mask]
        z_mid = (zb.left + zb.right) / 2
        predicted = norm_cdf(z_mid)
        actual = sub["outcome_up"].mean()
        rows.append({
            "z_bin": str(zb),
            "n": len(sub),
            "pred_up%": f"{predicted:.1%}",
            "actual_up%": f"{actual:.1%}",
            "gap": f"{(actual-predicted):+.1%}",
            "avg_sigma": f"{sub['sigma'].mean():.2e}",
            "avg_tau": f"{sub['tau'].mean():.0f}s",
        })
    
    print(pd.DataFrame(rows).to_string(index=False))
    
    # Distribution stats
    print(f"\n  z_raw stats: mean={sdf['z_raw'].mean():.3f}, std={sdf['z_raw'].std():.3f}, "
          f"skew={sdf['z_raw'].skew():.3f}, kurt={sdf['z_raw'].kurtosis():.3f}")
    print(f"  |z_raw| > 1.5: {(sdf['z_raw'].abs() > 1.5).mean():.1%} of observations")
    print(f"  |z_raw| > 3.0: {(sdf['z_raw'].abs() > 3.0).mean():.1%} of observations")
    print(f"  sigma stats: median={sdf['sigma'].median():.2e}, "
          f"p95={sdf['sigma'].quantile(0.95):.2e}, max={sdf['sigma'].max():.2e}")

for mk in MARKETS:
    z_score_analysis(signal_dfs[mk], label=MARKETS[mk]["label"])

## 4. Volatility Regime Impact

Key question: Does the GBM model break down during high-vol regimes? Do vol spikes cause systematically wrong predictions?

In [ ]:
def vol_regime_analysis(sdf, label=""):
    """Analyze model performance across vol regimes."""
    if sdf.empty:
        return
    
    sdf = sdf.copy()
    # Bin by sigma percentile
    sdf["vol_regime"] = pd.qcut(sdf["sigma"], q=4, labels=["Low", "Med-Low", "Med-High", "High"])
    
    print(f"\n{'='*70}")
    print(f"  VOLATILITY REGIME ANALYSIS: {label}")
    print(f"{'='*70}")
    
    rows = []
    for regime in ["Low", "Med-Low", "Med-High", "High"]:
        mask = sdf["vol_regime"] == regime
        if mask.sum() == 0:
            continue
        sub = sdf[mask]
        brier = ((sub["p_model"] - sub["outcome_up"]) ** 2).mean()
        
        # How often does the model "pick the right side"?
        # If p_model > 0.5, predicts UP; if < 0.5, predicts DOWN
        correct = ((sub["p_model"] > 0.5) & (sub["outcome_up"] == 1)) | \
                  ((sub["p_model"] < 0.5) & (sub["outcome_up"] == 0))
        accuracy = correct.mean()
        
        # How confident is the model? (distance from 0.5)
        confidence = (sub["p_model"] - 0.5).abs().mean()
        
        rows.append({
            "regime": regime,
            "n_obs": len(sub),
            "sigma_range": f"{sub['sigma'].min():.2e} - {sub['sigma'].max():.2e}",
            "brier": f"{brier:.4f}",
            "accuracy": f"{accuracy:.1%}",
            "avg_confidence": f"{confidence:.3f}",
            "actual_up%": f"{sub['outcome_up'].mean():.1%}",
        })
    
    print(pd.DataFrame(rows).to_string(index=False))
    
    # Tail analysis: extreme sigma
    p95 = sdf["sigma"].quantile(0.95)
    tail = sdf[sdf["sigma"] > p95]
    if len(tail) > 10:
        brier_tail = ((tail["p_model"] - tail["outcome_up"]) ** 2).mean()
        brier_rest = ((sdf[sdf["sigma"] <= p95]["p_model"] - sdf[sdf["sigma"] <= p95]["outcome_up"]) ** 2).mean()
        print(f"\n  Tail vol (>p95, sigma>{p95:.2e}): Brier={brier_tail:.4f} ({len(tail)} obs)")
        print(f"  Normal vol (<=p95):                 Brier={brier_rest:.4f}")
        print(f"  -> High-vol Brier is {'WORSE' if brier_tail > brier_rest else 'BETTER'} by {abs(brier_tail-brier_rest):.4f}")

for mk in MARKETS:
    vol_regime_analysis(signal_dfs[mk], label=MARKETS[mk]["label"])

## 5. Time Remaining (tau) Analysis

Does the model work better early or late in the window?

In [ ]:
def tau_analysis(sdf, window_s, label=""):
    """Analyze model performance across time-remaining buckets."""
    if sdf.empty:
        return
    
    sdf = sdf.copy()
    
    if window_s == 900:  # 15m
        tau_edges = [0, 120, 300, 600, 900]
        tau_labels = ["0-2m", "2-5m", "5-10m", "10-15m"]
    else:  # 5m
        tau_edges = [0, 60, 120, 200, 300]
        tau_labels = ["0-1m", "1-2m", "2-3.3m", "3.3-5m"]
    
    sdf["tau_bin"] = pd.cut(sdf["tau"], bins=tau_edges, labels=tau_labels)
    
    print(f"\n{'='*70}")
    print(f"  TAU (TIME REMAINING) ANALYSIS: {label}")
    print(f"{'='*70}")
    
    rows = []
    for tl in tau_labels:
        mask = sdf["tau_bin"] == tl
        if mask.sum() == 0:
            continue
        sub = sdf[mask]
        brier = ((sub["p_model"] - sub["outcome_up"]) ** 2).mean()
        correct = ((sub["p_model"] > 0.5) & (sub["outcome_up"] == 1)) | \
                  ((sub["p_model"] < 0.5) & (sub["outcome_up"] == 0))
        accuracy = correct.mean()
        confidence = (sub["p_model"] - 0.5).abs().mean()
        
        rows.append({
            "tau_bin": tl,
            "n_obs": len(sub),
            "brier": f"{brier:.4f}",
            "accuracy": f"{accuracy:.1%}",
            "avg_|p-0.5|": f"{confidence:.3f}",
            "actual_up%": f"{sub['outcome_up'].mean():.1%}",
        })
    
    print(pd.DataFrame(rows).to_string(index=False))

for mk in MARKETS:
    tau_analysis(signal_dfs[mk], MARKETS[mk]["window_s"], label=MARKETS[mk]["label"])

## 6. Out-of-Sample Walk-Forward Test

**Critical test**: We split windows chronologically (first 60% train, last 40% test). Train the calibration table on the train set, evaluate on the test set. This simulates "never having seen the data."

We also do a rolling walk-forward: use all data before window N to calibrate, test on window N.

In [ ]:
def walk_forward_test(windows, vol_lookback_s=90, max_z=1.5, train_frac=0.6, label=""):
    """Walk-forward out-of-sample test.
    
    For each test window, we use only past windows to build a calibration table,
    then evaluate on the current window. This is true out-of-sample.
    """
    n = len(windows)
    split_idx = int(n * train_frac)
    
    print(f"\n{'='*70}")
    print(f"  WALK-FORWARD OOS TEST: {label}")
    print(f"  Total windows: {n} | Train (first {train_frac:.0%}): {split_idx} | Test: {n - split_idx}")
    print(f"{'='*70}")
    
    # Method 1: Fixed train/test split
    # Build calibration from train set, test on test set
    train_obs = []  # (z_capped, tau, outcome)
    for df, outcome, start_px, _ in windows[:split_idx]:
        prices = df["chainlink_price"].tolist()
        ts_list = df["ts_ms"].tolist()
        for idx in range(vol_lookback_s, len(df), 30):
            row = df.iloc[idx]
            tau = row["time_remaining_s"]
            if tau <= 0:
                continue
            lo = max(0, idx - vol_lookback_s)
            sigma = compute_vol_deduped(prices[lo:idx+1], ts_list[lo:idx+1])
            if sigma <= 0:
                continue
            delta = (row["chainlink_price"] - start_px) / start_px
            z_raw = delta / (sigma * math.sqrt(tau))
            z_capped = max(-max_z, min(max_z, z_raw))
            train_obs.append((z_capped, tau, outcome))
    
    # Build calibration table from training data
    Z_BIN_WIDTH = 0.5
    TAU_EDGES = [0, 120, 300, 600, 900]
    
    cell_outcomes = defaultdict(list)
    for z_c, tau, out in train_obs:
        z_bin = round(z_c / Z_BIN_WIDTH) * Z_BIN_WIDTH
        tau_idx = 0
        for i in range(len(TAU_EDGES) - 1):
            if TAU_EDGES[i] <= tau < TAU_EDGES[i+1]:
                tau_idx = i
                break
        cell_outcomes[(z_bin, tau_idx)].append(out)
    
    cal_table = {}
    cal_counts = {}
    for key, outcomes in cell_outcomes.items():
        cal_table[key] = sum(outcomes) / len(outcomes)
        cal_counts[key] = len(outcomes)
    
    # Evaluate on test set
    test_records = []
    for df, outcome, start_px, final_px in windows[split_idx:]:
        prices = df["chainlink_price"].tolist()
        ts_list = df["ts_ms"].tolist()
        for idx in range(vol_lookback_s, len(df), 30):
            row = df.iloc[idx]
            tau = row["time_remaining_s"]
            if tau <= 0:
                continue
            lo = max(0, idx - vol_lookback_s)
            sigma = compute_vol_deduped(prices[lo:idx+1], ts_list[lo:idx+1])
            if sigma <= 0:
                continue
            delta = (row["chainlink_price"] - start_px) / start_px
            z_raw = delta / (sigma * math.sqrt(tau))
            z_capped = max(-max_z, min(max_z, z_raw))
            
            p_gbm = norm_cdf(z_capped)
            
            # Calibrated probability (fusion with GBM)
            z_bin = round(z_capped / Z_BIN_WIDTH) * Z_BIN_WIDTH
            tau_idx_v = 0
            for i in range(len(TAU_EDGES) - 1):
                if TAU_EDGES[i] <= tau < TAU_EDGES[i+1]:
                    tau_idx_v = i
                    break
            key = (z_bin, tau_idx_v)
            cal_prior = 100.0
            if key in cal_table and cal_counts.get(key, 0) >= 20:
                n_cal = cal_counts[key]
                w = n_cal / (n_cal + cal_prior)
                p_fused = w * cal_table[key] + (1 - w) * p_gbm
            else:
                p_fused = p_gbm
            
            test_records.append({
                "p_gbm": p_gbm,
                "p_fused": p_fused,
                "z_capped": z_capped,
                "sigma": sigma,
                "tau": tau,
                "outcome_up": outcome,
            })
    
    test_df = pd.DataFrame(test_records)
    if test_df.empty:
        print("  No test data.")
        return None
    
    # Compare GBM vs Fused on OOS data
    brier_gbm = ((test_df["p_gbm"] - test_df["outcome_up"]) ** 2).mean()
    brier_fused = ((test_df["p_fused"] - test_df["outcome_up"]) ** 2).mean()
    
    acc_gbm = (((test_df["p_gbm"] > 0.5) & (test_df["outcome_up"] == 1)) | \
               ((test_df["p_gbm"] < 0.5) & (test_df["outcome_up"] == 0))).mean()
    acc_fused = (((test_df["p_fused"] > 0.5) & (test_df["outcome_up"] == 1)) | \
                 ((test_df["p_fused"] < 0.5) & (test_df["outcome_up"] == 0))).mean()
    
    print(f"\n  Out-of-sample results ({len(test_df)} observations):")
    print(f"  {'Metric':<25s} {'Pure GBM':>12s} {'Fused (cal)':>12s}")
    print(f"  {'-'*49}")
    print(f"  {'Brier Score':<25s} {brier_gbm:>12.4f} {brier_fused:>12.4f}")
    print(f"  {'Direction Accuracy':<25s} {acc_gbm:>11.1%} {acc_fused:>11.1%}")
    
    if brier_fused < brier_gbm:
        print(f"\n  -> Calibration HELPS OOS: Brier improved by {brier_gbm - brier_fused:.4f}")
    else:
        print(f"\n  -> Calibration HURTS OOS: Brier worsened by {brier_fused - brier_gbm:.4f}")
        print(f"     This suggests the calibration table is overfitting to training data.")
    
    return test_df, brier_gbm, brier_fused, acc_gbm, acc_fused

oos_results = {}
for mk in MARKETS:
    result = walk_forward_test(all_windows[mk], label=MARKETS[mk]["label"])
    if result is not None:
        oos_results[mk] = result

## 7. Rolling Walk-Forward Backtest (Per-Window)

This is a more realistic OOS test: for each window, we use *only* all prior windows to build the calibration, then simulate a trading decision.

In [ ]:
def rolling_walk_forward_backtest(
    windows, window_s=900, vol_lookback_s=90, max_z=1.5,
    edge_threshold=0.04, early_edge_mult=0.4,
    kelly_fraction=0.25, max_bet_fraction=0.0125,
    initial_bankroll=10000.0, min_warmup_windows=30,
    maker_mode=True, label="",
):
    """Rolling walk-forward: for each window, use only past data to predict.
    
    This is the gold standard OOS test. No future information leaks.
    """
    bankroll = initial_bankroll
    results = []
    past_obs = []  # accumulated (z_capped, tau, outcome)
    
    for wi, (df, outcome, start_px, final_px) in enumerate(windows):
        # First, accumulate observations from this window for FUTURE use
        prices = df["chainlink_price"].tolist()
        ts_list = df["ts_ms"].tolist()
        window_obs = []
        for idx in range(vol_lookback_s, len(df), 30):
            row = df.iloc[idx]
            tau = row["time_remaining_s"]
            if tau <= 0:
                continue
            lo = max(0, idx - vol_lookback_s)
            sigma = compute_vol_deduped(prices[lo:idx+1], ts_list[lo:idx+1])
            if sigma <= 0:
                continue
            delta = (row["chainlink_price"] - start_px) / start_px
            z_raw = delta / (sigma * math.sqrt(tau))
            z_capped = max(-max_z, min(max_z, z_raw))
            window_obs.append((z_capped, tau, outcome))
        
        # Skip if not enough warmup windows
        if wi < min_warmup_windows:
            past_obs.extend(window_obs)
            continue
        
        # Build calibration from ONLY past observations
        Z_BIN_WIDTH = 0.5
        TAU_EDGES_local = [0, 120, 300, 600, 900] if window_s == 900 else [0, 60, 120, 200, 300]
        cell_outcomes = defaultdict(list)
        for z_c, tau_v, out in past_obs:
            z_bin = round(z_c / Z_BIN_WIDTH) * Z_BIN_WIDTH
            tau_idx = len(TAU_EDGES_local) - 2
            for i in range(len(TAU_EDGES_local) - 1):
                if TAU_EDGES_local[i] <= tau_v < TAU_EDGES_local[i+1]:
                    tau_idx = i
                    break
            cell_outcomes[(z_bin, tau_idx)].append(out)
        
        cal = {k: sum(v)/len(v) for k, v in cell_outcomes.items()}
        cal_n = {k: len(v) for k, v in cell_outcomes.items()}
        
        # Now simulate a single decision point for this window
        # Pick a snapshot around the middle of the window (after warmup)
        # This mimics when the live bot would typically enter
        target_tau = window_s * 0.5  # trade at ~50% of window
        best_idx = None
        best_dist = float('inf')
        for idx in range(vol_lookback_s, len(df)):
            tau = df.iloc[idx]["time_remaining_s"]
            dist = abs(tau - target_tau)
            if dist < best_dist:
                best_dist = dist
                best_idx = idx
        
        if best_idx is None:
            past_obs.extend(window_obs)
            continue
        
        row = df.iloc[best_idx]
        tau = row["time_remaining_s"]
        if tau <= 0:
            past_obs.extend(window_obs)
            continue
        
        lo = max(0, best_idx - vol_lookback_s)
        sigma = compute_vol_deduped(prices[lo:best_idx+1], ts_list[lo:best_idx+1])
        if sigma <= 0:
            past_obs.extend(window_obs)
            continue
        
        current_px = row["chainlink_price"]
        delta = (current_px - start_px) / start_px
        z_raw = delta / (sigma * math.sqrt(tau))
        z_capped = max(-max_z, min(max_z, z_raw))
        p_gbm = norm_cdf(z_capped)
        
        # Fused probability
        z_bin = round(z_capped / Z_BIN_WIDTH) * Z_BIN_WIDTH
        tau_idx_v = len(TAU_EDGES_local) - 2
        for i in range(len(TAU_EDGES_local) - 1):
            if TAU_EDGES_local[i] <= tau < TAU_EDGES_local[i+1]:
                tau_idx_v = i
                break
        key = (z_bin, tau_idx_v)
        cal_prior = 100.0
        if key in cal and cal_n.get(key, 0) >= 20:
            w = cal_n[key] / (cal_n[key] + cal_prior)
            p_model = w * cal[key] + (1 - w) * p_gbm
        else:
            p_model = p_gbm
        
        # Get market prices from snapshot
        bid_up = row.get("best_bid_up")
        ask_up = row.get("best_ask_up")
        bid_down = row.get("best_bid_down")
        ask_down = row.get("best_ask_down")
        
        if pd.isna(bid_up) or pd.isna(ask_up) or pd.isna(bid_down) or pd.isna(ask_down):
            past_obs.extend(window_obs)
            continue
        if bid_up <= 0 or ask_up <= 0 or bid_down <= 0 or ask_down <= 0:
            past_obs.extend(window_obs)
            continue
        
        # Dynamic edge threshold
        dyn_threshold = edge_threshold * (1.0 + early_edge_mult * math.sqrt(tau / window_s))
        
        # Cost basis (maker: use bid, 0% fee)
        if maker_mode:
            cost_up = float(bid_up)
            cost_down = float(bid_down)
        else:
            cost_up = float(ask_up) + poly_fee(float(ask_up))
            cost_down = float(ask_down) + poly_fee(float(ask_down))
        
        edge_up = p_model - cost_up
        edge_down = (1.0 - p_model) - cost_down
        
        # Decision
        action = "FLAT"
        edge = 0.0
        entry_price = 0.0
        p_side = 0.5
        
        if edge_up > dyn_threshold and edge_up >= edge_down:
            action = "BUY_UP"
            edge = edge_up
            entry_price = cost_up
            p_side = p_model
        elif edge_down > dyn_threshold and edge_down > edge_up:
            action = "BUY_DOWN"
            edge = edge_down
            entry_price = cost_down
            p_side = 1.0 - p_model
        
        if action != "FLAT" and entry_price > 0 and entry_price < 1:
            # Size with Kelly
            kelly_f = max(0, (p_side - entry_price) / (1.0 - entry_price))
            frac = min(kelly_fraction * kelly_f, max_bet_fraction)
            size_usd = bankroll * frac
            if size_usd < 5 * entry_price:
                size_usd = min(5 * entry_price, bankroll * 0.02)
            
            shares = size_usd / entry_price
            cost = shares * entry_price
            
            # Resolve
            won = (action == "BUY_UP" and outcome == 1) or (action == "BUY_DOWN" and outcome == 0)
            payout = shares if won else 0
            pnl = payout - cost
            bankroll += pnl
            
            results.append({
                "window_idx": wi,
                "action": action,
                "edge": edge,
                "p_model": p_model,
                "p_gbm": p_gbm,
                "entry_price": entry_price,
                "shares": shares,
                "cost": cost,
                "payout": payout,
                "pnl": pnl,
                "won": won,
                "outcome_up": outcome,
                "sigma": sigma,
                "tau": tau,
                "z_capped": z_capped,
                "bankroll": bankroll,
                "dyn_threshold": dyn_threshold,
            })
        
        # Add this window's obs for future calibration
        past_obs.extend(window_obs)
    
    if not results:
        print(f"  {label}: No trades taken in walk-forward.")
        return None
    
    rdf = pd.DataFrame(results)
    n_trades = len(rdf)
    n_wins = rdf["won"].sum()
    win_rate = n_wins / n_trades
    total_pnl = rdf["pnl"].sum()
    avg_pnl = rdf["pnl"].mean()
    avg_edge = rdf["edge"].mean()
    
    # Sharpe
    periods_per_day = 96 if window_s == 900 else 288
    mean_r = rdf["pnl"].mean()
    std_r = rdf["pnl"].std()
    sharpe = (mean_r / std_r) * math.sqrt(periods_per_day) if std_r > 0 else 0
    
    # Max drawdown
    peak = initial_bankroll
    max_dd = 0
    for _, r in rdf.iterrows():
        if r["bankroll"] > peak:
            peak = r["bankroll"]
        dd = peak - r["bankroll"]
        if dd > max_dd:
            max_dd = dd
    
    print(f"\n{'='*70}")
    print(f"  ROLLING WALK-FORWARD BACKTEST: {label}")
    print(f"{'='*70}")
    print(f"  Trades:         {n_trades}")
    print(f"  Win rate:       {win_rate:.1%} ({n_wins}/{n_trades})")
    print(f"  Total PnL:      ${total_pnl:+.2f}")
    print(f"  Avg PnL/trade:  ${avg_pnl:+.2f}")
    print(f"  Avg edge:       {avg_edge:.4f}")
    print(f"  Max drawdown:   ${max_dd:.2f}")
    print(f"  Sharpe (ann.):  {sharpe:.2f}")
    print(f"  Final bankroll: ${bankroll:,.2f}")
    
    # Win rate by side
    for side in ["BUY_UP", "BUY_DOWN"]:
        sm = rdf[rdf["action"] == side]
        if len(sm) > 0:
            print(f"  {side:>10s}: {sm['won'].sum()}/{len(sm)} wins ({sm['won'].mean():.1%}), "
                  f"avg edge={sm['edge'].mean():.4f}, avg pnl=${sm['pnl'].mean():+.2f}")
    
    return rdf

wf_results = {}
for mk in MARKETS:
    rdf = rolling_walk_forward_backtest(
        all_windows[mk], window_s=MARKETS[mk]["window_s"],
        label=MARKETS[mk]["label"]
    )
    if rdf is not None:
        wf_results[mk] = rdf

## 8. Parameter Sensitivity (Edge Threshold Sweep)

Test different edge thresholds on the OOS data to find robust values. We do this on the LAST 40% of data using calibration from the FIRST 60%.

In [ ]:
def parameter_sweep(
    windows, window_s=900, vol_lookback_s=90, max_z=1.5,
    initial_bankroll=10000.0, min_warmup_windows=30, label="",
):
    """Sweep edge_threshold and early_edge_mult on walk-forward OOS basis."""
    
    edge_thresholds = [0.02, 0.03, 0.04, 0.05, 0.06, 0.08, 0.10, 0.12, 0.15]
    early_mults = [0.0, 0.2, 0.4, 0.6, 0.8]
    max_z_values = [1.0, 1.5, 2.0, 2.5]
    
    print(f"\n{'='*70}")
    print(f"  PARAMETER SWEEP (OOS Walk-Forward): {label}")
    print(f"{'='*70}")
    
    # First sweep: edge_threshold (fix early_mult=0.4, max_z=1.5)
    print(f"\n  --- Edge Threshold Sweep (early_mult=0.4, max_z=1.5) ---")
    sweep_rows = []
    for et in edge_thresholds:
        rdf = rolling_walk_forward_backtest(
            windows, window_s=window_s, vol_lookback_s=vol_lookback_s, max_z=1.5,
            edge_threshold=et, early_edge_mult=0.4,
            initial_bankroll=initial_bankroll, min_warmup_windows=min_warmup_windows,
            label="",
        )
        if rdf is not None and len(rdf) > 0:
            sweep_rows.append({
                "edge_thresh": et,
                "n_trades": len(rdf),
                "win_rate": f"{rdf['won'].mean():.1%}",
                "total_pnl": f"${rdf['pnl'].sum():+.2f}",
                "avg_edge": f"{rdf['edge'].mean():.4f}",
                "final_bank": f"${rdf['bankroll'].iloc[-1]:,.0f}",
            })
        else:
            sweep_rows.append({
                "edge_thresh": et,
                "n_trades": 0,
                "win_rate": "N/A",
                "total_pnl": "$0",
                "avg_edge": "N/A",
                "final_bank": f"${initial_bankroll:,.0f}",
            })
    print(pd.DataFrame(sweep_rows).to_string(index=False))
    
    # Second sweep: max_z (fix edge=0.04, early_mult=0.4)
    print(f"\n  --- Max Z Sweep (edge=0.04, early_mult=0.4) ---")
    z_rows = []
    for mz in max_z_values:
        rdf = rolling_walk_forward_backtest(
            windows, window_s=window_s, vol_lookback_s=vol_lookback_s, max_z=mz,
            edge_threshold=0.04, early_edge_mult=0.4,
            initial_bankroll=initial_bankroll, min_warmup_windows=min_warmup_windows,
            label="",
        )
        if rdf is not None and len(rdf) > 0:
            z_rows.append({
                "max_z": mz,
                "n_trades": len(rdf),
                "win_rate": f"{rdf['won'].mean():.1%}",
                "total_pnl": f"${rdf['pnl'].sum():+.2f}",
                "avg_edge": f"{rdf['edge'].mean():.4f}",
            })
        else:
            z_rows.append({"max_z": mz, "n_trades": 0, "win_rate": "N/A", "total_pnl": "$0", "avg_edge": "N/A"})
    print(pd.DataFrame(z_rows).to_string(index=False))
    
    # Third sweep: early_edge_mult (fix edge=0.04, max_z=1.5)
    print(f"\n  --- Early Edge Mult Sweep (edge=0.04, max_z=1.5) ---")
    em_rows = []
    for em in early_mults:
        rdf = rolling_walk_forward_backtest(
            windows, window_s=window_s, vol_lookback_s=vol_lookback_s, max_z=1.5,
            edge_threshold=0.04, early_edge_mult=em,
            initial_bankroll=initial_bankroll, min_warmup_windows=min_warmup_windows,
            label="",
        )
        if rdf is not None and len(rdf) > 0:
            em_rows.append({
                "early_mult": em,
                "n_trades": len(rdf),
                "win_rate": f"{rdf['won'].mean():.1%}",
                "total_pnl": f"${rdf['pnl'].sum():+.2f}",
                "avg_edge": f"{rdf['edge'].mean():.4f}",
            })
        else:
            em_rows.append({"early_mult": em, "n_trades": 0, "win_rate": "N/A", "total_pnl": "$0", "avg_edge": "N/A"})
    print(pd.DataFrame(em_rows).to_string(index=False))

# Run sweep on each market (suppress individual prints)
import io, sys
for mk in MARKETS:
    old_stdout = sys.stdout
    sys.stdout = buffer = io.StringIO()
    parameter_sweep(all_windows[mk], window_s=MARKETS[mk]["window_s"], label=MARKETS[mk]["label"])
    output = buffer.getvalue()
    sys.stdout = old_stdout
    # Only print the summary tables
    lines = output.split("\n")
    in_table = False
    for line in lines:
        if "===" in line or "---" in line or "edge_thresh" in line or "max_z" in line or "early_mult" in line or "SWEEP" in line:
            in_table = True
        if in_table:
            # Skip the rolling backtest detail prints
            if any(x in line for x in ["Trades:", "Win rate:", "Total PnL:", "Avg PnL", "Avg edge:", "Max drawdown:", "Sharpe", "Final bankroll:", "BUY_UP:", "BUY_DOWN:"]):
                continue
            if line.strip():
                print(line)

## 9. Heavy-Tail / Fat-Tail Analysis

GBM assumes log-normal returns (thin tails). Crypto has fat tails. Let's quantify how badly this affects us.

In [ ]:
def fat_tail_analysis(windows, window_s=900, vol_lookback_s=90, label=""):
    """Analyze the distribution of returns and compare GBM vs reality."""
    
    # Collect final deltas (window-level returns)
    deltas = []
    sigmas_at_mid = []  # sigma measured at midpoint of window
    
    for df, outcome, start_px, final_px in windows:
        delta_pct = (final_px - start_px) / start_px
        deltas.append(delta_pct)
        
        # Measure sigma at midpoint
        mid_idx = len(df) // 2
        lo = max(0, mid_idx - vol_lookback_s)
        prices = df["chainlink_price"].tolist()
        ts_list = df["ts_ms"].tolist()
        sigma = compute_vol_deduped(prices[lo:mid_idx+1], ts_list[lo:mid_idx+1])
        if sigma > 0:
            sigmas_at_mid.append(sigma)
    
    deltas = np.array(deltas)
    sigmas_at_mid = np.array(sigmas_at_mid)
    
    print(f"\n{'='*70}")
    print(f"  FAT TAIL ANALYSIS: {label}")
    print(f"{'='*70}")
    
    # Return distribution stats
    print(f"\n  Window returns (delta = (final - start) / start):")
    print(f"    N:       {len(deltas)}")
    print(f"    Mean:    {deltas.mean():.6f} ({deltas.mean()*100:.4f}%)")
    print(f"    Std:     {deltas.std():.6f} ({deltas.std()*100:.4f}%)")
    print(f"    Skew:    {pd.Series(deltas).skew():.3f}")
    print(f"    Kurt:    {pd.Series(deltas).kurtosis():.3f} (Normal=0, fat tails>0)")
    print(f"    Min:     {deltas.min():.6f} ({deltas.min()*100:.4f}%)")
    print(f"    Max:     {deltas.max():.6f} ({deltas.max()*100:.4f}%)")
    
    # Expected vs actual tail events
    if len(sigmas_at_mid) > 0:
        median_sigma = np.median(sigmas_at_mid)
        expected_sigma_window = median_sigma * math.sqrt(window_s)
        
        # Under GBM, returns should be N(0, sigma*sqrt(T))
        # Count tail events (beyond 2 sigma)
        normalized = deltas / expected_sigma_window if expected_sigma_window > 0 else deltas
        tail_2sig = (np.abs(normalized) > 2).mean()
        tail_3sig = (np.abs(normalized) > 3).mean()
        expected_2sig = 2 * (1 - norm_cdf(2))  # ~4.56%
        expected_3sig = 2 * (1 - norm_cdf(3))  # ~0.27%
        
        print(f"\n  Tail event comparison (using median sigma={median_sigma:.2e}):")
        print(f"    Expected sigma_window: {expected_sigma_window:.6f}")
        print(f"    >2-sigma events: actual={tail_2sig:.1%} vs GBM expected={expected_2sig:.1%} "
              f"(ratio={tail_2sig/expected_2sig:.1f}x)" if expected_2sig > 0 else "")
        print(f"    >3-sigma events: actual={tail_3sig:.1%} vs GBM expected={expected_3sig:.2%} "
              f"(ratio={tail_3sig/expected_3sig:.1f}x)" if expected_3sig > 0 else "")
        
        if tail_2sig > 2 * expected_2sig:
            print(f"    -> SIGNIFICANT fat tails detected! GBM underestimates extreme moves.")
        elif tail_2sig > 1.3 * expected_2sig:
            print(f"    -> Moderate fat tails. GBM is slightly optimistic about tail risk.")
        else:
            print(f"    -> Tail distribution is close to GBM expectations.")
    
    # Sigma distribution
    if len(sigmas_at_mid) > 0:
        print(f"\n  Volatility (sigma per second) at window midpoint:")
        print(f"    Median:  {np.median(sigmas_at_mid):.2e}")
        print(f"    Mean:    {np.mean(sigmas_at_mid):.2e}")
        print(f"    p90:     {np.percentile(sigmas_at_mid, 90):.2e}")
        print(f"    p95:     {np.percentile(sigmas_at_mid, 95):.2e}")
        print(f"    p99:     {np.percentile(sigmas_at_mid, 99):.2e}")
        print(f"    Max:     {np.max(sigmas_at_mid):.2e}")
        print(f"    CoV:     {np.std(sigmas_at_mid)/np.mean(sigmas_at_mid):.2f} (vol-of-vol)")

for mk in MARKETS:
    fat_tail_analysis(all_windows[mk], window_s=MARKETS[mk]["window_s"], label=MARKETS[mk]["label"])

## 10. Blind Prediction Test

"Assume we have never seen the data": For each window, using only the GBM model with NO calibration and NO tuning, make a prediction. Compare against actual outcomes.

In [ ]:
def blind_prediction_test(windows, window_s=900, vol_lookback_s=90, max_z=1.5, label=""):
    """Pure GBM prediction test - no calibration, no tuning.
    
    For each window, at 3 decision points (early/mid/late), make a pure GBM
    directional prediction and track accuracy.
    """
    decision_points = [
        ("Early (75% left)", 0.75),
        ("Mid (50% left)", 0.50),
        ("Late (25% left)", 0.25),
    ]
    
    print(f"\n{'='*70}")
    print(f"  BLIND PREDICTION TEST (Pure GBM, no tuning): {label}")
    print(f"{'='*70}")
    
    for dp_name, dp_frac in decision_points:
        target_tau = window_s * dp_frac
        predictions = []
        
        for df, outcome, start_px, final_px in windows:
            prices = df["chainlink_price"].tolist()
            ts_list = df["ts_ms"].tolist()
            
            # Find closest row to target tau
            best_idx = None
            best_dist = float('inf')
            for idx in range(vol_lookback_s, len(df)):
                tau = df.iloc[idx]["time_remaining_s"]
                dist = abs(tau - target_tau)
                if dist < best_dist:
                    best_dist = dist
                    best_idx = idx
            
            if best_idx is None or best_dist > 30:
                continue
            
            row = df.iloc[best_idx]
            tau = row["time_remaining_s"]
            if tau <= 0:
                continue
            
            lo = max(0, best_idx - vol_lookback_s)
            sigma = compute_vol_deduped(prices[lo:best_idx+1], ts_list[lo:best_idx+1])
            if sigma <= 0:
                continue
            
            current_px = row["chainlink_price"]
            delta = (current_px - start_px) / start_px
            z_raw = delta / (sigma * math.sqrt(tau))
            z_capped = max(-max_z, min(max_z, z_raw))
            p_up = norm_cdf(z_capped)
            
            predicted_up = p_up > 0.5
            correct = (predicted_up and outcome == 1) or (not predicted_up and outcome == 0)
            confidence = abs(p_up - 0.5)
            
            predictions.append({
                "p_up": p_up,
                "predicted_up": predicted_up,
                "outcome_up": outcome,
                "correct": correct,
                "confidence": confidence,
                "z": z_capped,
                "sigma": sigma,
            })
        
        pdf = pd.DataFrame(predictions)
        if pdf.empty:
            continue
        
        acc = pdf["correct"].mean()
        brier = ((pdf["p_up"] - pdf["outcome_up"]) ** 2).mean()
        
        # Accuracy by confidence level
        high_conf = pdf[pdf["confidence"] > 0.15]  # p_up > 0.65 or < 0.35
        med_conf = pdf[(pdf["confidence"] > 0.05) & (pdf["confidence"] <= 0.15)]
        low_conf = pdf[pdf["confidence"] <= 0.05]  # near 50/50
        
        print(f"\n  {dp_name}: {len(pdf)} predictions")
        print(f"    Overall accuracy: {acc:.1%} | Brier: {brier:.4f}")
        if len(high_conf) > 0:
            print(f"    High confidence (|p-0.5|>0.15): {high_conf['correct'].mean():.1%} "
                  f"({len(high_conf)} trades, avg conf={high_conf['confidence'].mean():.3f})")
        if len(med_conf) > 0:
            print(f"    Med  confidence (0.05<|p|<0.15): {med_conf['correct'].mean():.1%} "
                  f"({len(med_conf)} trades)")
        if len(low_conf) > 0:
            print(f"    Low  confidence (|p-0.5|<0.05):  {low_conf['correct'].mean():.1%} "
                  f"({len(low_conf)} trades) <- should be ~50%")

for mk in MARKETS:
    blind_prediction_test(all_windows[mk], window_s=MARKETS[mk]["window_s"], label=MARKETS[mk]["label"])

## 11. Market Microstructure Edge: UP vs DOWN Bias

In [ ]:
def market_bias_analysis(windows, label=""):
    """Analyze systematic biases in the market: UP overpricing, bid-ask asymmetry."""
    
    records = []
    for df, outcome, start_px, final_px in windows:
        # Sample at window midpoint
        mid_idx = len(df) // 2
        row = df.iloc[mid_idx]
        
        bid_up = row.get("best_bid_up")
        ask_up = row.get("best_ask_up")
        bid_down = row.get("best_bid_down")
        ask_down = row.get("best_ask_down")
        
        if any(pd.isna(x) for x in [bid_up, ask_up, bid_down, ask_down]):
            continue
        
        mid_up = (float(bid_up) + float(ask_up)) / 2
        mid_down = (float(bid_down) + float(ask_down)) / 2
        implied_sum = mid_up + mid_down
        
        records.append({
            "mid_up": mid_up,
            "mid_down": mid_down,
            "implied_sum": implied_sum,
            "outcome_up": outcome,
            "overround": implied_sum - 1.0,  # vigorish
            "up_premium": mid_up - 0.5,  # UP vs fair 50/50
        })
    
    if not records:
        return
    
    mdf = pd.DataFrame(records)
    
    print(f"\n{'='*70}")
    print(f"  MARKET BIAS ANALYSIS: {label}")
    print(f"{'='*70}")
    
    print(f"  Base rate: UP wins {mdf['outcome_up'].mean():.1%} of the time")
    print(f"  Avg market mid-UP:   {mdf['mid_up'].mean():.3f} (fair ~0.5)")
    print(f"  Avg market mid-DOWN: {mdf['mid_down'].mean():.3f}")
    print(f"  Avg overround (vig): {mdf['overround'].mean():.3f} (>0 = house edge)")
    print(f"  Avg UP premium:      {mdf['up_premium'].mean():+.3f}")
    
    # Is UP systematically overpriced?
    # Fair price = actual win rate. If mid_up > actual_up_rate, UP is overpriced.
    actual_up_rate = mdf['outcome_up'].mean()
    pricing_gap = mdf['mid_up'].mean() - actual_up_rate
    print(f"\n  UP pricing gap: market says {mdf['mid_up'].mean():.3f}, actual rate {actual_up_rate:.3f}")
    print(f"  -> UP is {'OVERPRICED' if pricing_gap > 0.01 else 'UNDERPRICED' if pricing_gap < -0.01 else 'FAIRLY PRICED'} "
          f"by {abs(pricing_gap):.3f}")
    
    if pricing_gap > 0.01:
        print(f"  -> This supports the down_edge_bonus parameter (optimism tax)")
        suggested_bonus = min(0.3, pricing_gap / mdf['mid_up'].mean())
        print(f"  -> Suggested down_edge_bonus: {suggested_bonus:.2f}")

for mk in MARKETS:
    market_bias_analysis(all_windows[mk], label=MARKETS[mk]["label"])

## 12. Vol-Adjusted Confidence Sizing

Test if varying position sizing by vol regime helps or hurts.

In [ ]:
def vol_adjusted_sizing_test(windows, window_s=900, vol_lookback_s=90, max_z=1.5, label=""):
    """Compare fixed sizing vs vol-adjusted sizing on walk-forward basis."""
    
    min_warmup = 30
    edge_threshold = 0.04
    early_edge_mult = 0.4
    bankroll_fixed = 10000.0
    bankroll_vol = 10000.0
    
    results_fixed = []
    results_vol = []
    
    # Collect all sigmas to compute median for vol-adjustment
    all_sigmas = []
    past_sigmas = []
    
    for wi, (df, outcome, start_px, final_px) in enumerate(windows):
        if wi < min_warmup:
            # Collect sigmas for baseline
            mid_idx = len(df) // 2
            lo = max(0, mid_idx - vol_lookback_s)
            prices = df["chainlink_price"].tolist()
            ts_list = df["ts_ms"].tolist()
            sigma = compute_vol_deduped(prices[lo:mid_idx+1], ts_list[lo:mid_idx+1])
            if sigma > 0:
                past_sigmas.append(sigma)
            continue
        
        # Get sigma at decision point
        target_tau = window_s * 0.5
        best_idx = None
        best_dist = float('inf')
        for idx in range(vol_lookback_s, len(df)):
            tau = df.iloc[idx]["time_remaining_s"]
            dist = abs(tau - target_tau)
            if dist < best_dist:
                best_dist = dist
                best_idx = idx
        
        if best_idx is None:
            continue
        
        row = df.iloc[best_idx]
        tau = row["time_remaining_s"]
        if tau <= 0:
            continue
        
        prices = df["chainlink_price"].tolist()
        ts_list = df["ts_ms"].tolist()
        lo = max(0, best_idx - vol_lookback_s)
        sigma = compute_vol_deduped(prices[lo:best_idx+1], ts_list[lo:best_idx+1])
        if sigma <= 0:
            continue
        
        current_px = row["chainlink_price"]
        delta = (current_px - start_px) / start_px
        z_raw = delta / (sigma * math.sqrt(tau))
        z_capped = max(-max_z, min(max_z, z_raw))
        p_up = norm_cdf(z_capped)
        
        bid_up = row.get("best_bid_up")
        bid_down = row.get("best_bid_down")
        if pd.isna(bid_up) or pd.isna(bid_down) or float(bid_up) <= 0 or float(bid_down) <= 0:
            past_sigmas.append(sigma)
            continue
        
        cost_up = float(bid_up)
        cost_down = float(bid_down)
        
        dyn_threshold = edge_threshold * (1 + early_edge_mult * math.sqrt(tau / window_s))
        edge_up = p_up - cost_up
        edge_down = (1 - p_up) - cost_down
        
        action = "FLAT"
        edge = 0
        entry_price = 0
        p_side = 0.5
        
        if edge_up > dyn_threshold and edge_up >= edge_down:
            action, edge, entry_price, p_side = "BUY_UP", edge_up, cost_up, p_up
        elif edge_down > dyn_threshold:
            action, edge, entry_price, p_side = "BUY_DOWN", edge_down, cost_down, 1 - p_up
        
        if action != "FLAT" and entry_price > 0 and entry_price < 1:
            kelly = max(0, (p_side - entry_price) / (1 - entry_price))
            base_frac = min(0.25 * kelly, 0.0125)
            
            # Vol adjustment: reduce size in high-vol, increase in low-vol
            median_sigma = np.median(past_sigmas) if past_sigmas else sigma
            vol_ratio = median_sigma / sigma if sigma > 0 else 1.0
            vol_ratio = max(0.3, min(2.0, vol_ratio))  # clamp
            vol_frac = min(base_frac * vol_ratio, 0.02)
            
            for bk, frac, res_list in [
                (bankroll_fixed, base_frac, results_fixed),
                (bankroll_vol, vol_frac, results_vol),
            ]:
                size = bk * frac
                if size < 5 * entry_price:
                    size = min(5 * entry_price, bk * 0.02)
                shares = size / entry_price
                cost = shares * entry_price
                won = (action == "BUY_UP" and outcome == 1) or (action == "BUY_DOWN" and outcome == 0)
                payout = shares if won else 0
                pnl = payout - cost
                res_list.append({"pnl": pnl, "won": won, "vol_ratio": vol_ratio})
            
            bankroll_fixed += results_fixed[-1]["pnl"]
            bankroll_vol += results_vol[-1]["pnl"]
        
        past_sigmas.append(sigma)
    
    if not results_fixed:
        return
    
    rf = pd.DataFrame(results_fixed)
    rv = pd.DataFrame(results_vol)
    
    print(f"\n{'='*70}")
    print(f"  VOL-ADJUSTED SIZING: {label}")
    print(f"{'='*70}")
    print(f"  {'':>20s} {'Fixed Size':>15s} {'Vol-Adjusted':>15s}")
    print(f"  {'Trades':<20s} {len(rf):>15d} {len(rv):>15d}")
    print(f"  {'Win Rate':<20s} {rf['won'].mean():>14.1%} {rv['won'].mean():>14.1%}")
    print(f"  {'Total PnL':<20s} ${rf['pnl'].sum():>13.2f} ${rv['pnl'].sum():>13.2f}")
    print(f"  {'Final Bankroll':<20s} ${bankroll_fixed:>13,.2f} ${bankroll_vol:>13,.2f}")

for mk in MARKETS:
    vol_adjusted_sizing_test(all_windows[mk], window_s=MARKETS[mk]["window_s"], label=MARKETS[mk]["label"])

## 13. Summary & Recommendations

In [ ]:
print("="*70)
print("  OVERALL SUMMARY & RECOMMENDATIONS")
print("="*70)

print("""
FINDINGS:

1. MODEL CALIBRATION
   The results above show how well p_model = Phi(z) matches actual outcomes.
   Check the Brier scores and ECE values:
   - Brier < 0.24 = better than random
   - ECE < 0.05 = reasonably calibrated

2. OUT-OF-SAMPLE PERFORMANCE
   The walk-forward test shows true OOS performance.
   Compare the GBM vs Fused Brier scores:
   - If Fused helps: calibration is adding value
   - If Fused hurts: calibration is overfitting

3. PARAMETER RECOMMENDATIONS
   Use the parameter sweep tables to find ROBUST parameters:
   - Look for parameters that give stable win rate across a RANGE
   - Avoid picking the single best — that's overfitting
   - A parameter is robust if neighbors give similar results

4. KEY RISK FACTORS
   - Fat tails: Crypto returns exceed GBM predictions
   - Vol spikes: model breaks down in extreme vol
   - UP bias: markets may systematically overprice UP

RECOMMENDATIONS:

a) edge_threshold: Use the value from the sweep where win rate is
   STABLE across neighboring values (not the highest win rate)

b) max_z: If lowering max_z improves OOS, the model is too confident
   at extremes. Consider 1.0-1.5 range.

c) Vol regime: If high-vol Brier >> low-vol Brier, the vol_kill_sigma
   and vol_regime_mult parameters are essential.

d) DOWN bias: If UP is overpriced, down_edge_bonus is justified.
   Use the suggested value from the market bias analysis.

e) Calibration: If OOS test shows calibration helps, keep it.
   If it hurts, rely on pure GBM (set cal_prior_strength very high).
""")